# Inverse Diagnostic Demo

Этот notebook использует только публичный API `run_inverse_diagnostic_cycle(...)` и notebook-loader без дублирования бизнес-логики.

In [ ]:
from pathlib import Path
import pandas as pd

from hidden_patterns_combat.app.inverse_diagnostic_cycle import run_inverse_diagnostic_cycle
from hidden_patterns_combat.ui.inverse_notebook import (
    display_inverse_plots,
    display_inverse_report,
    load_inverse_artifacts,
)


In [ ]:
repo_root = Path.cwd()
input_path = repo_root / "data" / "raw" / "episodes.xlsx"
base_output_dir = repo_root / "artifacts" / "inverse_diagnostic_runs"

isolated_run = True
cleanup_mode = "none"


In [ ]:
result = run_inverse_diagnostic_cycle(
    input_path=input_path,
    output_dir=base_output_dir,
    isolated_run=isolated_run,
    cleanup_mode=cleanup_mode,
    retrain=True,
    n_states=3,
    topology_mode="left_to_right",
    generate_plots=True,
    verbose=True,
)

final_output_dir = Path(result.final_output_dir)
artifacts = load_inverse_artifacts(
    final_output_dir,
    expected_run_id=result.run_id,
    expected_run_fingerprint=result.run_fingerprint,
)


## Current Run Metadata


In [ ]:
try:
    from IPython.display import Markdown, display

    manifest = artifacts.run_manifest or {}
    warning_lines = "\n".join(f"- {w}" for w in artifacts.loader_warnings[:20])
    if not warning_lines:
        warning_lines = "- none"

    display(Markdown(
        f"""
- run_id: `{result.run_id}`
- final_output_dir: `{result.final_output_dir}`
- run_manifest_path: `{result.run_manifest_path}`
- started_at: `{manifest.get('started_at', 'n/a')}`
- finished_at: `{manifest.get('finished_at', 'n/a')}`
- input_file: `{manifest.get('input_file_name', 'n/a')}`
- key warnings:
{warning_lines}
"""
    ))
except Exception:
    print("run_id:", result.run_id)
    print("final_output_dir:", result.final_output_dir)
    print("run_manifest_path:", result.run_manifest_path)
    print("started_at:", artifacts.run_manifest.get("started_at", "n/a"))
    print("finished_at:", artifacts.run_manifest.get("finished_at", "n/a"))
    print("input_file:", artifacts.run_manifest.get("input_file_name", "n/a"))
    print("warnings:")
    for warning in artifacts.loader_warnings[:20]:
        print("-", warning)


In [ ]:
artifacts.artifact_status


In [ ]:
display_inverse_report(artifacts.report_markdown)


In [ ]:
display_inverse_plots(artifacts.plot_paths)


In [ ]:
if not artifacts.episode_analysis.empty:
    preview_cols = [
        c
        for c in [
            "episode_id",
            "sequence_id",
            "observed_zap_class",
            "hidden_state_name",
            "confidence",
        ]
        if c in artifacts.episode_analysis.columns
    ]
    display(artifacts.episode_analysis[preview_cols].head(20))
else:
    print("episode_analysis.csv is missing or empty")
